# 09 Survival Analysis — Exercises

Practice survival analysis with the Pine and Cypress Nursing Home Legionella data.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# -- CJK font setup (avoids Chinese labels showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["death_date"] = pd.to_datetime(df["death_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

cases = df[df["infected"] == 1].copy()
cases["event"] = (cases["outcome"] == "dead").astype(int)

investigation_end = cases["symptom_onset_date"].max() + pd.Timedelta(days=14)
cases["end_date"] = cases.apply(
    lambda r: r["death_date"] if r["event"] == 1 else investigation_end, axis=1
)
cases["time_to_event"] = (cases["end_date"] - cases["symptom_onset_date"]).dt.days

## Question 1: Survival Analysis for CHF (Congestive Heart Failure)

1. Draw Kaplan-Meier survival curves for CHF (`comorbidity_chf`) present vs absent
2. Run a log-rank test
3. Does CHF significantly affect survival?

In [ ]:
# TODO: CHF vs no CHF KM curves
# TODO: Log-rank test
# TODO: Interpret

## Question 2: Survival Comparison by Age Group

1. Split the infected residents into two groups by age: `age >= 75` (elderly) vs `age < 75`
2. Draw the KM survival curves for the two groups
3. Run a log-rank test
4. Does age significantly affect survival?

In [ ]:
# TODO: Split into age >= 75 vs < 75
# TODO: KM curves
# TODO: Log-rank test
# TODO: Interpret

## Question 3 (Challenge): Survival Analysis of Hospitalized vs Not Hospitalized + Cox Regression

1. Draw KM survival curves for hospitalized (`hospitalized == 1`) vs not hospitalized
2. Run a log-rank test
3. Build a Cox regression model including: `age`, `is_male`, `hospitalized`, `comorbidity_copd`, `comorbidity_chf`
4. Draw the HR forest plot
5. Interpret: is hospitalization a "protective factor" or a "marker associated with severity"?

In [ ]:
# TODO: hospitalized vs not KM curves
# TODO: Log-rank test
# TODO: Cox regression (age, is_male, hospitalized, copd, chf)
# TODO: HR forest plot
# TODO: Interpret

## Question 4: Survival Analysis of TB Treatment (Tuberculosis Scenario)

1. Use `drug_resistant` (whether the patient has multidrug-resistant TB, MDR-TB) to split patients into two groups
2. Draw the Kaplan-Meier survival curves for the two groups reaching cure (event)
3. Run a log-rank test to compare whether time-to-cure differs significantly between the two groups
4. Calculate the median time to cure (median survival time) for each group
5. Interpret: does drug-resistant TB significantly delay time to cure? What does this mean for clinical care and health policy?

In [ ]:
# Data: TB treatment cohort -- compare time to cure between drug-resistant (MDR-TB) and non-resistant patients
rng = np.random.default_rng(409)
n = 400

drug_resistant = rng.binomial(1, 0.2, size=n)  # 20% have multidrug-resistant TB (MDR-TB)
age = np.clip(rng.normal(48, 16, size=n), 15, 90).round().astype(int)

# Non-resistant patients are cured in about 150 days on average; resistant patients need longer treatment, averaging about 320 days to cure
scale_cure = np.where(drug_resistant == 1, 320, 150)
duration_to_cure = rng.exponential(scale_cure)

follow_up_end = 540  # 18-month follow-up; patients still not cured beyond this are right-censored
time_to_event = np.minimum(duration_to_cure, follow_up_end)
event = (duration_to_cure <= follow_up_end).astype(int)  # 1 = cured, 0 = still not cured at end of follow-up (censored)

tb = pd.DataFrame({
    "patient_id": [f"TB{i:04d}" for i in range(n)],
    "age": age,
    "drug_resistant": drug_resistant,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# TODO: Draw KM curves for drug_resistant vs non-resistant patients (event = cure)
# TODO: Run a log-rank test
# TODO: Calculate the median time to cure (median survival time) for each group
# TODO: Interpret: does drug-resistant TB significantly delay time to cure? What does this mean for clinical care and health policy?

## Question 5: COVID-19 Hospitalization Survival Analysis (ICU Grouping)

1. Use `icu_admission` to split hospitalized patients into ICU and general ward groups
2. Draw Kaplan-Meier curves for survival during hospitalization (event = death) for the two groups
3. Run a log-rank test
4. Build a Cox regression model with covariates: `age`, `is_male`, `icu_admission`, `diabetes`
5. Interpret: does the hazard ratio (HR) for ICU admission mean ICU care itself increases the risk of death? Or does it reflect that patients admitted to the ICU were already more severely ill (confounding by indication)?

In [ ]:
# Data: COVID-19 hospitalization cohort -- compare time from admission to death between ICU and general ward patients
rng = np.random.default_rng(519)
n = 500

age = np.clip(rng.normal(60, 18, size=n), 18, 95).round().astype(int)
is_male = rng.binomial(1, 0.5, size=n)
icu_admission = rng.binomial(1, 0.22, size=n)
diabetes = rng.binomial(1, 0.25, size=n)

baseline_hazard = 1 / 420  # Baseline risk of death for a 60-year-old, non-diabetic, general ward patient
linear_pred = 1.0 * icu_admission + 0.4 * diabetes + 0.03 * (age - 60)
hazard = baseline_hazard * np.exp(linear_pred)
duration = rng.exponential(1 / hazard)

follow_up_end = 60  # 60-day follow-up; patients still hospitalized or discharged beyond this are right-censored
time_to_event = np.minimum(duration, follow_up_end)
event = (duration <= follow_up_end).astype(int)  # 1 = died during hospitalization, 0 = discharged or end of follow-up (censored)

covid = pd.DataFrame({
    "patient_id": [f"CV{i:04d}" for i in range(n)],
    "age": age,
    "is_male": is_male,
    "icu_admission": icu_admission,
    "diabetes": diabetes,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# TODO: Draw KM survival curves for ICU vs general ward patients (event = death during hospitalization)
# TODO: Run a log-rank test
# TODO: Build a Cox regression model with covariates age, is_male, icu_admission, diabetes
# TODO: Interpret the hazard ratio (HR) for icu_admission -- does this mean ICU care itself is harmful, or that only the most severely ill patients are admitted to the ICU?

## Question 6: Survival Analysis of Measles Contact Follow-up (Measles Scenario)

1. Use `vaccinated` (whether the contact was previously vaccinated against measles) to split contacts into two groups
2. Draw Kaplan-Meier curves for developing symptoms (event) within a 21-day observation period for the two groups
3. Run a log-rank test
4. Calculate the median time to onset for the unvaccinated group (this approximates the median incubation period of measles)
5. Interpret: does vaccination significantly reduce or delay onset? What does this mean for contact tracing and quarantine period policy?

In [ ]:
# Data: measles case contacts -- compare time from exposure to onset between vaccinated and unvaccinated contacts
rng = np.random.default_rng(626)
n = 350

vaccinated = rng.binomial(1, 0.6, size=n)  # 60% of contacts were previously vaccinated against measles
age = np.clip(rng.normal(10, 8, size=n), 0, 60).round().astype(int)

# Unvaccinated contacts have a high probability of infection (high attack rate); most vaccinated contacts are protected, so breakthrough infection is rare
p_infected = np.where(vaccinated == 1, 0.12, 0.85)
infected = rng.binomial(1, p_infected)

# If infected, the incubation period (exposure to onset) is about 10-14 days; uninfected contacts do not develop symptoms during the observation period
incubation = np.clip(rng.normal(12, 2.2, size=n), 5, 21)
surveillance_end = 21  # Contacts are followed for 21 days

time_to_event = np.where(infected == 1, incubation, surveillance_end)
event = infected  # 1 = developed symptoms within observation period, 0 = no symptoms by end of observation (censored)

measles = pd.DataFrame({
    "contact_id": [f"MS{i:04d}" for i in range(n)],
    "age": age,
    "vaccinated": vaccinated,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# TODO: Draw KM curves for vaccinated vs unvaccinated contacts (event = onset within the 21-day observation period)
# TODO: Run a log-rank test
# TODO: Calculate the median time to onset for the unvaccinated group (this approximates the median incubation period of measles)
# TODO: Interpret: does vaccination significantly reduce or delay onset? What does this mean for contact tracing and quarantine period policy?

## Question 7: Survival Analysis of Dengue Progression to Severe Disease (Dengue Scenario)

1. Use `secondary_infection` (whether the patient had a secondary infection) to split patients into two groups
2. Draw Kaplan-Meier curves for progression to severe dengue (event) within a 14-day clinical follow-up period for the two groups
3. Run a log-rank test
4. Build a Cox regression model with covariates: `age`, `is_male`, `secondary_infection`
5. Interpret: does the hazard ratio (HR) for secondary infection significantly increase the risk of progression to severe disease? Is this consistent with the mechanism of antibody-dependent enhancement (ADE)?

In [ ]:
# Data: dengue case cohort -- compare time to progression to severe disease between secondary and primary infection patients
rng = np.random.default_rng(727)
n = 450

secondary_infection = rng.binomial(1, 0.35, size=n)  # 35% had a secondary infection (previously infected with a different dengue serotype)
age = np.clip(rng.normal(32, 16, size=n), 1, 85).round().astype(int)
is_male = rng.binomial(1, 0.48, size=n)

baseline_hazard = 1 / 45  # Baseline risk of progression to severe disease for a 32-year-old primary infection patient
linear_pred = 1.1 * secondary_infection + 0.012 * (age - 32)
hazard = baseline_hazard * np.exp(linear_pred)
duration = rng.exponential(1 / hazard)

follow_up_end = 14  # 14-day clinical follow-up period after onset
time_to_event = np.minimum(duration, follow_up_end)
event = (duration <= follow_up_end).astype(int)  # 1 = progressed to severe dengue, 0 = did not progress by end of follow-up (censored)

dengue = pd.DataFrame({
    "case_id": [f"DF{i:04d}" for i in range(n)],
    "age": age,
    "is_male": is_male,
    "secondary_infection": secondary_infection,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# TODO: Draw KM curves for secondary vs primary infection patients (event = progression to severe dengue)
# TODO: Run a log-rank test
# TODO: Build a Cox regression model with covariates age, is_male, secondary_infection
# TODO: Interpret: does the hazard ratio (HR) for secondary infection significantly increase the risk of progression to severe disease? Is this consistent with the mechanism of antibody-dependent enhancement (ADE)?

## Question 8 (Challenge): Multivariable Cox Regression and Proportional Hazards Assumption Check (Legionella Scenario)

Continue using the Pine and Cypress Nursing Home `cases` loaded at the start of this chapter (infected cases, `event` = death, `time_to_event` = days from onset to death or end of follow-up).

1. Build a Cox regression model with covariates: `age`, `is_male`, `icu_admission`, `immunosuppressed`, `comorbidity_cancer`
2. Print the hazard ratio (HR) and 95% confidence interval for each variable
3. Use `cph.check_assumptions()` to check whether the proportional hazards assumption holds
4. Calculate the model's concordance index (C-index) to assess its predictive discrimination
5. Interpret: which factors significantly increase the risk of death? Is the proportional hazards assumption violated? What does the C-index say about the model's discrimination?

In [ ]:
# Data: continue using cases loaded at the start of this chapter (Pine and Cypress Nursing Home Legionella cases, event = death)
cox_cols_q8 = ["time_to_event", "event", "age", "sex",
               "icu_admission", "immunosuppressed", "comorbidity_cancer"]
cox_df8 = cases[cox_cols_q8].copy()
cox_df8["is_male"] = (cox_df8["sex"] == "M").astype(int)
cox_df8 = cox_df8.drop(columns=["sex"])

print(cox_df8.describe().round(2))

# TODO: Build a Cox regression model (age, is_male, icu_admission, immunosuppressed, comorbidity_cancer)
# TODO: Print HR and 95% confidence interval
# TODO: Use cph.check_assumptions() to check the proportional hazards assumption
# TODO: Calculate the C-index (cph.concordance_index_)
# TODO: Interpret